In [1]:
import pysam
import pandas as pd
import random


In [33]:
tumor = "t24"
variant_list = pd.read_csv(f"/mnt/wigstore6/user/ranade/binless_clustering/DNA_alone/{tumor}/tumor_specific_variants.txt", sep="\t", header = None)

In [34]:
variant_list.head()

,0,1,2,3,4
0,chr10,502176,502177,G,A
1,chr10,502976,502977,T,C
2,chr10,813429,813430,G,C
3,chr10,884804,884805,G,A
4,chr10,1238424,1238425,T,C


In [35]:
fastafilename = "/mnt/wigstore6/user/ranade/LOH_analysis/reference/hg19.fa"
fasta = pysam.FastaFile(fastafilename)

In [36]:




chrom = "chr10"
pos = 205975       # 1-based
ref = "A"
window = 20

start = max(1, pos - window)
end   = pos + window

seq = fasta.fetch(chrom, start - 1, end)  # 0-based, half-open

# scan for matching reference bases
candidates = []
for i, base in enumerate(seq):
    candidate_pos = start + i
    #print(candidate_pos, base)
    if candidate_pos == pos:
        continue
    if base == ref or base == ref.lower():
        candidates.append(candidate_pos)

In [37]:
candidates

[205955,
 205958,
 205962,
 205965,
 205966,
 205968,
 205969,
 205973,
 205977,
 205981,
 205984,
 205990,
 205991]

In [38]:
def pick_parallel_candidate(chrom, pos, ref, window = 50):
    start = max(1, pos - window)
    end   = pos + window

    seq = fasta.fetch(chrom, start - 1, end)  # 0-based, half-open

    # scan for matching reference bases
    candidates = []
    for i, base in enumerate(seq):
        candidate_pos = start + i
        #print(candidate_pos, base)
        if candidate_pos == pos:
            continue
        if base == ref or base == ref.lower():
            candidates.append(candidate_pos)
    return random.choice(candidates)

In [39]:
#pick a random one from candidates

random.choice(candidates)


205977

In [40]:
 row_list = []
 for row in variant_list.values:
    chrom = row[0]
    pos = row[2]
    ref = row[3]
    alt = row[4]
    try:
      parallel_pos = pick_parallel_candidate(chrom, pos, ref, window = 50)
    except:
      parallel_pos = pick_parallel_candidate(chrom, pos, ref, window = 200)
    row_list.append( (chrom, parallel_pos-1, parallel_pos, ref, alt) )




In [41]:
pd.DataFrame(row_list).to_csv(f"/mnt/wigstore6/user/ranade/binless_clustering/DNA_alone/{tumor}/tumor_specific_variants_parallel_positions.txt", sep="\t", header = None, index = False)